# Subtask 3: Dimensional Aspect Sentiment Quadruple Prediction (DimASQP)
Extract (Aspect, Category, Opinion, VA) quadruples from text.

**Input**: Text only
**Output**: List of (aspect, category, opinion, valence#arousal) quadruples

In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate sentencepiece

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import re
from typing import List, Dict, Tuple
from collections import Counter
import warnings
import shutil
warnings.filterwarnings('ignore')

2026-02-08 16:42:46.363964: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770568966.746367      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770568966.850895      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770568967.765329      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568967.765375      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568967.765378      55 computation_placer.cc:177] computation placer alr

In [3]:
# Configuration
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimasqp/subtask3'
    DATA_DIR = './subtask3_data'
    OUTPUT_DIR = './subtask3_outputs'
    CHECKPOINT_DIR = './subtask3_checkpoints'
    
    # Model
    MODEL_NAME = 't5-base'
    MAX_INPUT_LENGTH = 128
    MAX_OUTPUT_LENGTH = 320
    
    # Training
    BATCH_SIZE = 12
    EPOCHS = 20
    LEARNING_RATE = 3e-4
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    GRADIENT_ACCUMULATION_STEPS = 2
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    
    # Decoding
    NUM_BEAMS = 4
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42
    
    # Valid categories (from data analysis)
    VALID_CATEGORIES = [
        'FOOD#QUALITY', 'SERVICE#GENERAL', 'RESTAURANT#GENERAL',
        'AMBIENCE#GENERAL', 'FOOD#STYLE_OPTIONS', 'RESTAURANT#MISCELLANEOUS',
        'FOOD#PRICES', 'RESTAURANT#PRICES', 'DRINKS#QUALITY',
        'DRINKS#STYLE_OPTIONS', 'LOCATION#GENERAL', 'DRINKS#PRICES',
        'FOOD#GENERAL', 'SERVICE#QUALITY'
    ]

config = Config()
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
files_to_copy = ["eng_laptop_dev_task3.jsonl", "eng_laptop_test_task3.jsonl",
            "eng_laptop_train_alltasks.jsonl", "eng_restaurant_dev_task3.jsonl",
            "eng_restaurant_test_task3.jsonl", "eng_restaurant_train_alltasks.jsonl"]
TRAIN_ZIP = '/kaggle/input/dimasqp/subtask3'
DATA_DIR = './subtask3_data'
for fname in files_to_copy:
    src = os.path.join(TRAIN_ZIP, fname)
    dst = os.path.join(DATA_DIR, fname)
    shutil.copy(src, dst)
    print(f"Copied {fname} → {dst}")
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

print(f"Device: {config.DEVICE}")
print(f"Valid categories: {len(config.VALID_CATEGORIES)}")

Copied eng_laptop_dev_task3.jsonl → ./subtask3_data/eng_laptop_dev_task3.jsonl
Copied eng_laptop_test_task3.jsonl → ./subtask3_data/eng_laptop_test_task3.jsonl
Copied eng_laptop_train_alltasks.jsonl → ./subtask3_data/eng_laptop_train_alltasks.jsonl
Copied eng_restaurant_dev_task3.jsonl → ./subtask3_data/eng_restaurant_dev_task3.jsonl
Copied eng_restaurant_test_task3.jsonl → ./subtask3_data/eng_restaurant_test_task3.jsonl
Copied eng_restaurant_train_alltasks.jsonl → ./subtask3_data/eng_restaurant_train_alltasks.jsonl
Device: cuda
Valid categories: 14


In [4]:
# Data loading functions
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def parse_va(va_string):
    """Parse VA string 'V.VV#A.AA' to (valence, arousal)"""
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    """Format VA to 'V.VV#A.AA'"""
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

def validate_category(category):
    """Map generated category to closest valid category"""
    if category in config.VALID_CATEGORIES:
        return category
    
    # Simple fuzzy matching
    category_upper = category.upper()
    for valid_cat in config.VALID_CATEGORIES:
        if category_upper in valid_cat or valid_cat in category_upper:
            return valid_cat
    
    # Default fallback based on keywords
    if 'FOOD' in category_upper:
        return 'FOOD#QUALITY'
    elif 'DRINK' in category_upper:
        return 'DRINKS#QUALITY'
    elif 'SERVICE' in category_upper:
        return 'SERVICE#GENERAL'
    elif 'AMBIENCE' in category_upper or 'ATMOSPHERE' in category_upper:
        return 'AMBIENCE#GENERAL'
    else:
        return 'RESTAURANT#GENERAL'

In [5]:
# Format conversion for T5 seq2seq
def quadruples_to_sequence(quadruples):
    """
    Convert quadruple list to sequence format for T5.
    Format: [A]aspect[C]category[O]opinion[V]valence#arousal[SEP]...
    """
    if not quadruples:
        return "none"
    
    parts = []
    for quad in quadruples:
        aspect = quad['Aspect']
        category = quad['Category']
        opinion = quad['Opinion']
        va = quad['VA']
        parts.append(f"[A]{aspect}[C]{category}[O]{opinion}[V]{va}")
    
    return "[SEP]".join(parts)

def sequence_to_quadruples(sequence):
    """
    Parse T5 output sequence back to quadruple list.
    """
    if sequence.strip() == "none":
        return []
    
    quadruples = []
    pattern = r'\[A\](.*?)\[C\](.*?)\[O\](.*?)\[V\](.*?)(?:\[SEP\]|$)'
    matches = re.finditer(pattern, sequence)
    
    for match in matches:
        aspect = match.group(1).strip()
        category = match.group(2).strip()
        opinion = match.group(3).strip()
        va = match.group(4).strip()
        
        # Validate category
        category = validate_category(category)
        
        # Validate VA format
        if '#' in va:
            try:
                v, a = parse_va(va)
                va = format_va(v, a)
                quadruples.append({
                    'Aspect': aspect,
                    'Category': category,
                    'Opinion': opinion,
                    'VA': va
                })
            except:
                continue
    
    return quadruples

In [6]:
# Dataset class
class DimASQPDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length, max_output_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['Text']
        quadruples = item.get('Quadruplet', [])
        
        # Format: "extract quadruples: <text>"
        input_text = f"extract quadruples: {text}"
        target_text = quadruples_to_sequence(quadruples)
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_output_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'labels': labels,
            'id': item['ID'],
            'text': text
        }

# Test dataset (no labels)
class DimASQPTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['Text']
        
        input_text = f"extract quadruples: {text}"
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'id': item['ID'],
            'text': text
        }

In [7]:
# Training function with gradient accumulation
def train_epoch(model, dataloader, optimizer, scheduler, device, accumulation_steps):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(dataloader, desc='Training')
    for idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / accumulation_steps
        loss.backward()
        
        if (idx + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += outputs.loss.item()
        progress_bar.set_postfix({'loss': outputs.loss.item()})
    
    return total_loss / len(dataloader)

In [8]:
# Evaluation function
def evaluate(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            total_loss += outputs.loss.item()
    
    return total_loss / len(dataloader)

In [9]:
# Prediction function
def predict(model, dataloader, tokenizer, device):
    model.eval()
    predictions = {}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            ids = batch['id']
            
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            
            for id_, pred_text in zip(ids, decoded):
                quadruples = sequence_to_quadruples(pred_text)
                predictions[id_] = quadruples
    
    return predictions

def save_predictions(predictions, original_data, output_path):
    """Save predictions in JSONL format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in original_data:
            id_ = item['ID']
            output = {
                'ID': id_,
                'Quadruplet': predictions.get(id_, [])
            }
            f.write(json.dumps(output) + '\n')
    print(f"Predictions saved to {output_path}")

In [10]:
# Load data
print("Loading data...")
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')
dev_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task3.jsonl')
dev_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task3.jsonl')

# Combine domains
train_data = train_rest + train_laptop
dev_data = dev_rest + dev_laptop

print(f"Train samples: {len(train_data)}")
print(f"Dev samples: {len(dev_data)}")

# Initialize tokenizer
tokenizer = T5Tokenizer.from_pretrained(config.MODEL_NAME)

# Create datasets
train_dataset = DimASQPDataset(
    train_data, tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)
dev_dataset = DimASQPDataset(
    dev_data, tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=config.BATCH_SIZE)

Loading data...
Train samples: 6360
Dev samples: 400


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [11]:
# Initialize model
print("Initializing model...")
model = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME).to(config.DEVICE)

# Optimizer and scheduler
optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader) * config.EPOCHS // config.GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

Initializing model...


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Total training steps: 5300
Warmup steps: 530


In [13]:
# Training loop
best_loss = float('inf')
patience = 3  # Stop if no improvement for 3 epochs
patience_counter = 0
history = {'train_loss': [], 'dev_loss': []}

print("\nStarting training...")
for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch + 1}/{config.EPOCHS}")
    
    train_loss = train_epoch(
        model, train_loader, optimizer, scheduler,
        config.DEVICE, config.GRADIENT_ACCUMULATION_STEPS
    )
    dev_loss = evaluate(model, dev_loader, tokenizer, config.DEVICE)
    
    history['train_loss'].append(train_loss)
    history['dev_loss'].append(dev_loss)
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Dev Loss: {dev_loss:.4f}")
    
    # Save best model
    if dev_loss < best_loss:
        best_loss = dev_loss
        patience_counter = 0  # Reset counter
        model.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        tokenizer.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        print(f"✓ Saved best model (Loss: {dev_loss:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{patience})")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch + 1}")
            break

print(f"\nTraining completed. Best loss: {best_loss:.4f}")


Starting training...

Epoch 1/20


Training:   0%|          | 0/530 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]

Train Loss: 1.4037
Dev Loss: 0.4270
✓ Saved best model (Loss: 0.4270)

Epoch 2/20


Training:   0%|          | 0/530 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]

Train Loss: 0.2374
Dev Loss: 0.3924
No improvement (2/3)

Epoch 6/20


Training:   0%|          | 0/530 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]

Train Loss: 0.2192
Dev Loss: 0.3966
No improvement (3/3)

Early stopping at epoch 6

Training completed. Best loss: 0.3670


In [14]:
# Load best model for inference
print("Loading best model...")
model = T5ForConditionalGeneration.from_pretrained(
    f'{config.CHECKPOINT_DIR}/best_model'
).to(config.DEVICE)
tokenizer = T5Tokenizer.from_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
print("Model loaded successfully")

Loading best model...
Model loaded successfully


In [16]:
# Generate predictions for test sets
print("\nGenerating predictions for test sets...")

# Restaurant test
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task3.jsonl')
test_rest_dataset = DimASQPTestDataset(test_rest_data, tokenizer, config.MAX_INPUT_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, tokenizer, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

# Laptop test
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task3.jsonl')
test_laptop_dataset = DimASQPTestDataset(test_laptop_data, tokenizer, config.MAX_INPUT_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, tokenizer, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\nAll predictions completed!")


Generating predictions for test sets...


Predicting:   0%|          | 0/84 [00:00<?, ?it/s]

Predictions saved to ./subtask3_outputs/restaurant_test_predictions.jsonl


Predicting:   0%|          | 0/84 [00:00<?, ?it/s]

Predictions saved to ./subtask3_outputs/laptop_test_predictions.jsonl

All predictions completed!


In [17]:
# Sample predictions
print("\nSample predictions:")
with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(json.dumps(json.loads(line), indent=2))


Sample predictions:
{
  "ID": "rest26_asqp_test_1",
  "Quadruplet": [
    {
      "Aspect": "stewed chicken",
      "Category": "FOOD#QUALITY",
      "Opinion": "excellent",
      "VA": "8.00#8.00"
    },
    {
      "Aspect": "curried chicken",
      "Category": "FOOD#QUALITY",
      "Opinion": "excellent",
      "VA": "8.00#8.00"
    }
  ]
}
{
  "ID": "rest26_asqp_test_2",
  "Quadruplet": [
    {
      "Aspect": "service",
      "Category": "SERVICE#GENERAL",
      "Opinion": "absolutely horrendous",
      "VA": "2.50#7.00"
    }
  ]
}
{
  "ID": "rest26_asqp_test_3",
  "Quadruplet": [
    {
      "Aspect": "service",
      "Category": "SERVICE#GENERAL",
      "Opinion": "very nice",
      "VA": "6.50#6.33"
    },
    {
      "Aspect": "service",
      "Category": "SERVICE#GENERAL",
      "Opinion": "slow",
      "VA": "3.67#6.33"
    }
  ]
}


In [18]:
# Category validation check
print("\nCategory distribution in predictions:")
all_categories = []
with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        for quad in data.get('Quadruplet', []):
            all_categories.append(quad['Category'])

category_counts = Counter(all_categories)
print(f"Total quadruples: {len(all_categories)}")
print(f"Unique categories: {len(category_counts)}")
for cat, count in category_counts.most_common(10):
    print(f"  {cat}: {count}")


Category distribution in predictions:
Total quadruples: 2131
Unique categories: 12
  FOOD#QUALITY: 1071
  SERVICE#GENERAL: 549
  AMBIENCE#GENERAL: 138
  RESTAURANT#GENERAL: 88
  FOOD#PRICES: 81
  DRINKS#QUALITY: 57
  FOOD#STYLE_OPTIONS: 38
  RESTAURANT#PRICES: 36
  RESTAURANT#MISCELLANEOUS: 27
  DRINKS#STYLE_OPTIONS: 22
